In [ ]:
import itertools as it

import boto3
import botocore
import numpy as np
import pandas as pd
from pandas.util import hash_pandas_object
from scipy import stats
import seaborn as sns
import seaborn.objects as so
from teeplot import teeplot as tp


In [ ]:
from dishpylib.pyhelpers import make_outattr_metadata
from dishpylib.pyhelpers import print_runtime


In [ ]:
print_runtime()


In [ ]:
teeplot_subdir = "2025-12-10-complexty-fitness-correlation"


# get data


In [ ]:
s3_handle = boto3.resource(
    's3',
    region_name="us-east-2",
    config=botocore.config.Config(
        signature_version=botocore.UNSIGNED,
    ),
)
bucket_handle = s3_handle.Bucket('prq49')

series_profiles, = bucket_handle.objects.filter(
    Prefix=f'endeavor=16/series-profiles/stage=8+what=elaborated/',
)


In [ ]:
df = pd.read_csv(
    f's3://prq49/{series_profiles.key}',
    compression='xz',
)
dfdigest = '{:x}'.format( hash_pandas_object( df ).sum() )
dfdigest


In [ ]:
def make_outattr_metadata():
    return {}


In [ ]:
for stint in df['Stint'].unique():
    exec(f'df{stint} = df[ df["Stint"] == {stint} ]')


In [ ]:
dfm10 = df[ df['Stint'] % 10 == 0 ]


In [ ]:
s3_handle = boto3.resource(
    "s3",
    region_name="us-east-2",
    config=botocore.config.Config(
        signature_version=botocore.UNSIGNED,
    ),
)
bucket_handle = s3_handle.Bucket("prq49")

dfabio = pd.concat(
    [
        pd.read_csv(f"s3://prq49/{item.key}")
        for item in bucket_handle.objects.filter(
            Prefix=f"endeavor=16/external-competitions/stage=2+what=collated/",
        )
    ],
    ignore_index=True,
)
dfabio["kind"] = "abio"


In [ ]:
s3_handle = boto3.resource(
    "s3",
    region_name="us-east-2",
    config=botocore.config.Config(
        signature_version=botocore.UNSIGNED,
    ),
)
bucket_handle = s3_handle.Bucket("prq49")

dfbio = pd.concat(
    [
        pd.read_csv(f"s3://prq49/{item.key}")
        for item in bucket_handle.objects.filter(
            Prefix=f"endeavor=16/external-competitions-focalbb/stage=2+what=collated/",
        )
    ],
    ignore_index=True,
)
dfbio["kind"] = "bio"


In [ ]:
dfx = pd.concat([dfabio, dfbio], ignore_index=True)
dfx["Fitness Differential Focal Sign"] = np.sign(
    dfx["Fitness Differential Focal"],
)
dfx["Stint"] = dfx["Competition Stint"]
dfx["Series"] = dfx["genome series"]
dfx = dfx[
    dfx["Root ID"] == 1
].groupby(["Stint", "Series", "kind"]).agg(
    {
        "Focal Prevalence": ["mean", "median"],
    },
)
dfx.columns = [f"{col[0]} {col[1].title()}" for col in dfx.columns.values]
dfx


In [ ]:
dfj = dfx.join(
    dfm10[
        [
            "Stint",
            "Series",
            "Fitness Complexity",
            "Flagged Advantageous Sites",
            "Flagged Deleterious Sites",
        ]
    ].set_index(
        ["Stint", "Series"]
    ),
    on=["Stint", "Series"],
    how="inner",
).reset_index(drop=False)
dfj["Favored Mean"] = dfj["Focal Prevalence Mean"] > 0.5
dfj["Favored Median"] = dfj["Focal Prevalence Median"] > 0.5
dfj


In [ ]:
for hue, kind in it.product(
    ["Favored Mean", "Favored Median"],
    ["abio", "bio"],
):
    dfjx = dfj[dfj["kind"] == kind].astype({
        "Stint": "str",
    })
    with tp.teed(
        sns.violinplot,
        data=dfjx.replace({True: "More Fit", False: "Less Fit"}),
        x="Stint",
        y="Flagged Advantageous Sites",
        gap=0.2,
        hue=hue,
        inner=None,
        split=True,
        teeplot_outattrs={"kind": kind},
        teeplot_subdir=teeplot_subdir,
    ) as ax:
        ax.figure.set_size_inches(6, 1.2)
        sns.despine(ax=ax)
        sns.stripplot(
            data=dfjx.replace({True: "More Fit", False: "Less Fit"}),
            x="Stint",
            y="Flagged Advantageous Sites",
            hue=hue,
            hue_order=["Less Fit", None, None, None, "More Fit"],
            alpha=0.5,
            ax=ax,
            dodge=True,
            jitter=0.6,
            legend=False,
            palette=[
                sns.color_palette("dark")[0],
                "red",
                "red",
                "red",
                sns.color_palette("dark")[1],
            ],
            size=2,
        )
        (
            so.Plot(
                data=dfjx.replace({True: "More Fit", False: "Less Fit"}),
                x="Stint",
                y="Flagged Advantageous Sites",
                color=hue,
            )
            .add(
                so.Range(
                    linewidth=2.5,
                ),
                so.Est("mean", seed=1),
                so.Dodge(gap=0.1),
                legend=False,
            )
            .scale(
                color=so.Nominal(
                    [*it.repeat("gainsboro", 100)],
                    order=[None, "Less Fit", "More Fit", None],
                ),
            )
            .on(ax)
            .plot()
        )
        (
            so.Plot(
                data=dfjx.replace({True: "More Fit", False: "Less Fit"}),
                x="Stint",
                y="Flagged Advantageous Sites",
                color=hue,
            )
            .add(
                so.Dot(artist_kws=dict(zorder=15), marker="_", pointsize=3),
                so.Agg("mean"),
                so.Dodge(gap=0.1),
                legend=False,
            )
            .scale(
                color=so.Nominal(
                    [*it.repeat("black", 4)],
                    order=[None, "Less Fit", "More Fit", None],
                ),
            )
            .on(ax)
            .plot()
        )
        sns.move_legend(
            ax,
            "upper left",
            bbox_to_anchor=(0.5, 1.1),
            ncols=2,
            frameon=False,
            title=None,
        )
        ax.set_ylabel("Genetic\nComplexity")


In [ ]:
for y, kind in it.product(
    ["Focal Prevalence Mean", "Focal Prevalence Median"],
    ["abio", "bio"],
):
    dfjx = dfj[dfj["kind"] == kind].astype({
        "Stint": "str",
    })
    with tp.teed(
        sns.lmplot,
        data=dfj,
        col="Stint",
        x="Flagged Advantageous Sites",
        y=y,
        facet_kws=dict(
            sharex=False,
            sharey=True,
        ),
        scatter_kws={"alpha": 0.5, "clip_on": False, "s": 3},
        teeplot_outattrs={"kind": kind},
        teeplot_subdir=teeplot_subdir,
    ) as g:
        g.figure.set_size_inches(9, 2)
        g.set_xlabels("Genetic Complexity")
        g.set_ylabels("Fitness")
        g.set_titles("Stint {col_name}")
        g.set(ylim=(-0.02, 1.02))
        for i, ax in enumerate(g.axes.flat):
            if i != 4:
                ax.set_xlabel(None)
